# Notebook 07: Forward pass of the final model

We will build this model with plain Python lists:

```text
5 inputs -> N ReLU hidden neurons -> 1 output
```

## What this forward pass will do

This notebook follows one input example through a tiny neural network.

1. Start with 5 input numbers.
2. Build `hidden_raw`, one raw value for each hidden neuron.
3. Apply ReLU to get `hidden_after_relu`.
4. Use the hidden values later to make one final `prediction`.

The values `hidden_raw` and `hidden_after_relu` are cache values. A cache value is an intermediate result we save because the backward pass will need it later.


For the first derivation, use a tiny hidden layer:

```text
5 inputs -> 3 ReLU hidden neurons -> 1 output
```

| Name | Shape when `hidden_size = 3` | Meaning |
|---|---:|---|
| `inputs` | 5 numbers | One input example |
| `input_to_hidden_weights` | 3 rows × 5 numbers | One row per hidden neuron |
| `hidden_biases` | 3 numbers | One bias per hidden neuron |
| `hidden_raw` | 3 numbers | Hidden values before ReLU |
| `hidden_after_relu` | 3 numbers | Hidden values after ReLU |
| `hidden_to_output_weights` | 3 numbers | One output weight per hidden neuron |
| `output_bias` | 1 number | Final output bias |
| `prediction` | 1 number | The model’s final number |

## Step 1: deterministic initialization

The model needs starting weights and biases before it can make a prediction. A **weight** is a number on a connection, and a **bias** is an extra number added to a neuron.

We use `random.Random(seed)` so the values look random but repeat every time this notebook runs. That makes the lesson easier to debug because the same seed creates the same starting model.


In [6]:
import random

input_count = 5
hidden_size = 3
seed = 7

rng = random.Random(seed)
INPUTS = [0.2, -0.4, 0.1, 0.7, -0.3]

INPUT_TO_HIDDEN_WEIGHTS: list[list[float]] = [
    [round(rng.uniform(-0.5, 0.5), 3) for _ in range(input_count)]
    for _ in range(hidden_size)
]

HIDDEN_BIASES: list[float] = [
    round(rng.uniform(-0.5, 0.5), 3) for _ in range(hidden_size)
]

HIDDEN_TO_OUTPUT_WEIGHTS: list[float] = [
    round(rng.uniform(-0.5, 0.5), 3) for _ in range(hidden_size)
]

OUTPUT_BIAS: float = round(rng.uniform(-0.5, 0.5), 3)

INPUT_TO_HIDDEN_WEIGHTS, HIDDEN_BIASES, HIDDEN_TO_OUTPUT_WEIGHTS, OUTPUT_BIAS

([[-0.176, -0.349, 0.151, -0.428, 0.036],
  [-0.134, -0.442, 0.007, -0.463, -0.066],
  [-0.43, -0.409, -0.075, 0.327, -0.376]],
 [-0.277, 0.127, 0.448],
 [0.077, -0.103, 0.476],
 -0.453)

## Step 2: compute the hidden raw values

Each hidden neuron combines all 5 inputs into one number. For one neuron, the rule is:

```text
raw value = bias + input_1 × weight_1 + input_2 × weight_2 + ... + input_5 × weight_5
```

`INPUT_TO_HIDDEN_WEIGHTS` has one row per hidden neuron. The loop below picks one row, pairs each input with one weight, multiplies each pair, adds those products, and then adds the neuron bias.


In [7]:
hidden_raw = []

for neuron_index in range(hidden_size):
    weights_for_neuron = INPUT_TO_HIDDEN_WEIGHTS[neuron_index]
    bias_for_neuron = HIDDEN_BIASES[neuron_index]

    total = bias_for_neuron
    for input_value, weight in zip(INPUTS, weights_for_neuron):
        total += weight * input_value

    hidden_raw.append(round(total, 3))

hidden_raw

[-0.468, -0.027, 0.86]

## Checkpoint: trace one hidden neuron by hand

For hidden neuron 0, the code is computing this value:

```text
-0.277
+ (0.2 × -0.176)
+ (-0.4 × -0.349)
+ (0.1 × 0.151)
+ (0.7 × -0.428)
+ (-0.3 × 0.036)
= -0.468 after rounding
```

This is a good place to pause and check the shape idea: 5 inputs need 5 matching weights for this one neuron.


## Step 3: apply ReLU

ReLU is a simple gate:

```text
ReLU(x) = max(0, x)
```

If a hidden raw value is negative, ReLU turns it into `0.0`. If a hidden raw value is positive, ReLU keeps it. This gives us `hidden_after_relu`, which is the hidden layer output that will feed the final prediction.


In [8]:
hidden_after_relu = []

for raw_value in hidden_raw:
    hidden_after_relu.append(max(0.0, raw_value))

hidden_after_relu

[0.0, 0.0, 0.86]

## Cache values saved so far

The forward pass has now saved two important hidden-layer values:

| Cache name | What it stores | Why it matters later |
|---|---|---|
| `hidden_raw` | The hidden neuron values before ReLU | Backpropagation uses these to know which ReLU gates were open |
| `hidden_after_relu` | The hidden neuron values after ReLU | The output neuron uses these to compute the final prediction |

The next step is to combine `hidden_after_relu`, `HIDDEN_TO_OUTPUT_WEIGHTS`, and `OUTPUT_BIAS` into one numeric `prediction`.
